# Notebook 07: Verify & Audit Dataset V2 & Huấn Luyện Thực Nghiệm Trên Google Colab

> **Mục tiêu của Notebook:**
> 1. **Giai đoạn 1 (Audit & Verification):** Quét đệ quy tìm các file zip dataset-v2, kiểm tra checksum SHA256, khảo sát cấu trúc tệp bên trong ZIP, kiểm tra nhãn, tính toàn vẹn (pairing, leakage) và tự động sinh file `data.yaml` chuẩn cho YOLO11.
> 2. **Giai đoạn 2 (Training & Evaluation):** Cho phép chọn nhanh tỉ lệ split (ví dụ: `70_15_15`), giải nén siêu tốc lên SSD Colab và chạy thực nghiệm Baseline + CBAM (hỗ trợ lưu checkpoint lên Drive và tự khôi phục khi bị ngắt kết nối/crash).

---

## 📌 Bước 1️⃣: Kết Nối Google Drive & Chuẩn Bị Môi Trường
> **Yêu cầu trước khi chạy:**
> - Tạo lối tắt (Shortcut) của thư mục chứa dataset hoặc thư mục chung `KLTN_38` / `KLTN-2026-testingNtraining` vào **My Drive** của bạn.

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Tự động nhận diện thư mục dataset-v2 trên Drive
POSSIBLE_DATASET_PATHS = [
    "/content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/dataset-v2",
    "/content/drive/MyDrive/KLTN_38/dataset/dataset-v2",
    "/content/drive/MyDrive/dataset/dataset-v2",
    "/content/drive/MyDrive/dataset-v2"
]

DATASET_V2_DIR = None
for p in POSSIBLE_DATASET_PATHS:
    if os.path.exists(p):
        DATASET_V2_DIR = p
        break

if DATASET_V2_DIR is None:
    # Tìm kiếm đệ quy nhanh trong MyDrive nếu chưa thấy
    my_drive = Path("/content/drive/MyDrive")
    found = list(my_drive.rglob("dataset-v2"))
    if found:
        DATASET_V2_DIR = str(found[0])

if DATASET_V2_DIR is None:
    print("⚠️ Chưa tự động tìm thấy thư mục dataset-v2 theo đường dẫn mặc định.")
    print("👉 Hãy gán thủ công DATASET_V2_DIR nếu bạn đặt tên khác.")
    DATASET_V2_DIR = "/content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/dataset-v2"
else:
    print(f"✅ Đã tìm thấy thư mục Dataset V2 tại: {DATASET_V2_DIR}")

# Clone repo mã nguồn
!git clone https://github.com/mizzhau/yolo11n-cbam-mvtec-defect-detection.git /content/klcn2026
%cd /content/klcn2026
!git checkout main
!git pull origin main

# Cài đặt thư viện
!pip install -q ultralytics albumentations tabulate pyyaml
print("\n🚀 Môi trường đã sẵn sàng!")

## 🔍 Bước 2️⃣: GIAI ĐOẠN 1 — Kiểm Tra Toàn Vẹn SHA256 & Khảo Sát Tệp ZIP
> Cell này sẽ quét tìm cả 3 file zip (`80_10_10`, `70_15_15`, `60_20_20`) để:
> 1. Tính toán mã băm SHA256 thực tế của file `.zip` và đối chiếu với file `.sha256` đi kèm.
> 2. Đọc lướt danh sách tệp bên trong ZIP để xác định cấu trúc (đường dẫn ảnh, nhãn, tỉ lệ phân chia train/val/test).

In [ ]:
import hashlib
import zipfile
import os
from pathlib import Path
from collections import Counter
from tabulate import tabulate

assert DATASET_V2_DIR and os.path.exists(DATASET_V2_DIR), f"❌ Thư mục không tồn tại: {DATASET_V2_DIR}"
BASE = Path(DATASET_V2_DIR)

SPLITS = {"80_10_10": (80, 10, 10), "70_15_15": (70, 15, 15), "60_20_20": (60, 20, 20)}

def find_zip(key):
    hits = list(BASE.rglob(f"KLTN_prep_v2_{key}.zip"))
    return hits[0] if hits else None

def sha256(p, chunk=1<<24):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()

def split_of(name):
    # Nhận diện split qua folder parts hoặc tên file
    parts = Path(name).parts
    for s in ("train", "val", "test"):
        if s in parts:
            return s
    # Nếu file phẳng ở dạng train_xxx.png hoặc trong category/train/...
    lower_name = name.lower()
    for s in ("train", "val", "test"):
        if f"/{s}/" in f"/{lower_name}" or f"_{s}_" in lower_name:
            return s
    return None

rows, structs = [], {}
found_zips = {}

print("=" * 80)
print("🔍 BẮT ĐẦU KIỂM TRA SHA256 VÀ KHẢO SÁT FILE ZIP TRÊN GOOGLE DRIVE")
print("=" * 80)

for key, ratio in SPLITS.items():
    zp = find_zip(key)
    if zp is None:
        rows.append([key, "MISSING", "N/A", "N/A", "❌ Không tìm thấy zip"])
        continue

    found_zips[key] = str(zp)
    shap = zp.with_name(zp.name + ".sha256")
    exp = shap.read_text(encoding="utf-8").split()[0].lower() if shap.exists() else None
    act = sha256(zp).lower()
    sha_st = "⚠️ NO .sha256" if exp is None else ("✅ KHỚP" if act == exp else "❌ LỆCH")

    with zipfile.ZipFile(zp) as z:
        if z.testzip():
            rows.append([key, "CORRUPT", act[:8], "-", "❌ File zip bị hỏng"])
            continue
        names = [n for n in z.namelist() if not n.endswith("/")]

    imgs = [n for n in names if n.lower().endswith((".jpg", ".jpeg", ".png"))]
    lbls = [n for n in names if n.endswith(".txt") and not n.endswith("data.yaml")]
    cnt = Counter(split_of(n) for n in imgs)
    tot = sum(cnt.values()) or 1
    actual = tuple(round(100 * cnt.get(s, 0) / tot, 1) for s in ("train", "val", "test"))
    ratio_ok = all(abs(a - e) < 1.5 for a, e in zip(actual, ratio))

    stems = lambda L: {Path(n).stem for n in L}
    missing_lbl = len(stems(imgs) - stems(lbls))
    leak = {s: {Path(n).stem for n in imgs if split_of(n) == s} for s in ("train", "val", "test")}
    overlap = (len(leak.get("train", set()) & leak.get("val", set())) +
               len(leak.get("train", set()) & leak.get("test", set())) +
               len(leak.get("val", set()) & leak.get("test", set())))

    rows.append([key, zp.name, f"{sha_st} ({act[:8]})",
                 f"{dict(cnt)} → {actual}% {'✅' if ratio_ok else '⚠️'}",
                 f"thiếu nhãn: {missing_lbl} | trùng split: {overlap}"])
    structs[key] = names[:10]

print(tabulate(rows, headers=["Tỉ lệ", "Zip", "SHA256", "Số ảnh / tỉ lệ thực", "Pairing / Leakage"], tablefmt="grid"))

print("\n📁 10 File mẫu đầu tiên trong mỗi zip:")
for k, samples in structs.items():
    print(f"\n-- [{k}] --")
    for sample in samples:
        print(f"   {sample}")

## 🧪 Bước 3️⃣: Giải Nén Thử Nghiệm & Audit Chi Tiết Nhãn
> Chọn 1 bộ dataset (mặc định: `70_15_15`) để giải nén lên ổ cứng SSD cục bộ của Colab (`/content/dataset_v2/`).
> Sau đó chạy script kiểm tra:
> - Tọa độ Bounding Box `[x_center, y_center, width, height]` có nằm chuẩn trong `[0, 1]`.
> - Class ID trong nhãn có hợp lệ (nằm trong `0..47`).

In [ ]:
import os
import yaml
from glob import glob
from pathlib import Path

# Chọn tỉ lệ muốn verify & dùng để huấn luyện:
SELECTED_RATIO = "70_15_15"  # Tùy chọn: '80_10_10', '70_15_15', '60_20_20'

# Lấy file zip đã tìm thấy ở Bước 2 hoặc rglob lại
TARGET_ZIP = found_zips.get(SELECTED_RATIO) if 'found_zips' in locals() and SELECTED_RATIO in found_zips else None
if TARGET_ZIP is None:
    hits = list(Path(DATASET_V2_DIR).rglob(f"KLTN_prep_v2_{SELECTED_RATIO}.zip"))
    assert hits, f"❌ Không tìm thấy zip cho {SELECTED_RATIO}"
    TARGET_ZIP = str(hits[0])

EXTRACT_DIR = f"/content/dataset_v2_{SELECTED_RATIO}"

print(f"📦 Đang giải nén {TARGET_ZIP} vào {EXTRACT_DIR}...")
!mkdir -p "{EXTRACT_DIR}"
!unzip -q "{TARGET_ZIP}" -d "{EXTRACT_DIR}"
print(f"✅ Giải nén hoàn tất vào {EXTRACT_DIR}!")

print("\n📂 Cấu trúc thư mục mức 1:")
!ls -la "{EXTRACT_DIR}"

all_label_files = glob(f"{EXTRACT_DIR}/**/*.txt", recursive=True)
all_label_files = [f for f in all_label_files if not f.endswith("data.yaml") and not f.endswith(".sha256")]

print(f"\n🔍 Tìm thấy tổng cộng {len(all_label_files)} file nhãn txt.")

invalid_classes = set()
invalid_bboxes = 0
total_objects = 0

for lf in all_label_files:
    with open(lf, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                total_objects += 1
                cls_id = int(parts[0])
                if cls_id < 0 or cls_id >= 48:
                    invalid_classes.add(cls_id)
                coords = [float(x) for x in parts[1:5]]
                if any(c < 0.0 or c > 1.0 for c in coords):
                    invalid_bboxes += 1

print(f"📊 Tổng số bounding boxes: {total_objects}")
if invalid_classes:
    print(f"❌ CẢNH BÁO: Phát hiện Class ID nằm ngoài dải 0..47: {invalid_classes}")
else:
    print("✅ Toàn bộ Class ID đều hợp lệ trong khoảng 0..47!")

if invalid_bboxes > 0:
    print(f"⚠️ Có {invalid_bboxes} bbox có tọa độ vượt ngưỡng [0, 1].")
else:
    print("✅ Toàn bộ tọa độ Bounding Box đều chuẩn hóa đúng trong [0, 1]!")

## 🛠️ Bước 4️⃣: Tự Động Tạo / Chuẩn Hóa File `data.yaml` Tuyệt Đối
> Tự động tìm thư mục `images/{train,val,test}` hoặc danh sách `.txt` trong dữ liệu vừa giải nén để sinh file `configs/data/exp_2/data_colab_active.yaml` chuẩn xác cho YOLO.

In [ ]:
import yaml
from pathlib import Path

REF_YAML = "/content/klcn2026/configs/data/exp_2/data_70_15_15.yaml"
with open(REF_YAML, "r", encoding="utf-8") as f:
    ref_data = yaml.safe_load(f)
class_names = ref_data.get("names", [])

extract_p = Path(EXTRACT_DIR)

# Tự động phát hiện cấu trúc images
if (extract_p / "images" / "train").exists():
    train_path = str(extract_p / "images" / "train")
    val_path = str(extract_p / "images" / "val")
    test_path = str(extract_p / "images" / "test") if (extract_p / "images" / "test").exists() else val_path
elif list(extract_p.glob("*train*.txt")):
    train_txt = list(extract_p.glob("*train*.txt"))[0]
    val_txt = list(extract_p.glob("*val*.txt"))[0]
    test_txt_list = list(extract_p.glob("*test*.txt"))
    test_txt = test_txt_list[0] if test_txt_list else val_txt
    train_path = str(train_txt)
    val_path = str(val_txt)
    test_path = str(test_txt)
elif any((p / "images" / "train").exists() for p in extract_p.iterdir() if p.is_dir()):
    sub_p = [p for p in extract_p.iterdir() if (p / "images" / "train").exists()][0]
    train_path = str(sub_p / "images" / "train")
    val_path = str(sub_p / "images" / "val")
    test_path = str(sub_p / "images" / "test") if (sub_p / "images" / "test").exists() else val_path
else:
    train_path = str(extract_p)
    val_path = str(extract_p)
    test_path = str(extract_p)

colab_yaml_content = {
    "path": EXTRACT_DIR,
    "train": train_path,
    "val": val_path,
    "test": test_path,
    "nc": len(class_names),
    "names": class_names
}

ACTIVE_YAML_PATH = "/content/klcn2026/configs/data/exp_2/data_colab_active.yaml"
with open(ACTIVE_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(colab_yaml_content, f, allow_unicode=True, sort_keys=False)

print(f"✅ Đã tạo file YAML cấu hình chuẩn tại: {ACTIVE_YAML_PATH}")
print("\n--- Nội dung YAML ---")
!cat {ACTIVE_YAML_PATH}

## 🚀 Bước 5️⃣: GIAI ĐOẠN 2 — Thực Nghiệm Huấn Luyện (Baseline & CBAM Variants)
> Cấu hình thư mục lưu kết quả trên Google Drive để không bị mất khi hết phiên Colab.

In [ ]:
RUNS_DIR = f"/content/drive/MyDrive/KLTN-2026-testingNtraining/runs_exp2_{SELECTED_RATIO}"
os.makedirs(f"{RUNS_DIR}/baseline", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/cbam_backbone", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/cbam_neck", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/cbam_head", exist_ok=True)
os.makedirs(f"{RUNS_DIR}/cbam_backbone_head", exist_ok=True)

print(f"✅ Thư mục lưu kết quả trên Drive: {RUNS_DIR}")

### Cell 5A: Huấn Luyện YOLO11n BASELINE (Exp 0)

In [ ]:
!python src/training/train_baseline.py \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}/baseline" \
    --name train

### Cell 5B: Huấn Luyện CBAM BACKBONE (Exp 1)

In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_backbone.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}/cbam_backbone" \
    --name train

### Cell 5C: Huấn Luyện CBAM NECK (Exp 2)

In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_neck.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}/cbam_neck" \
    --name train

### Cell 5D: Huấn Luyện CBAM HEAD (Exp 4)

In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_head.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}/cbam_head" \
    --name train

### Cell 5E: Huấn Luyện CBAM BACKBONE + HEAD (Biến Thể Phối Hợp)

In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_backbone_head.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --project "{RUNS_DIR}/cbam_backbone_head" \
    --name train

### 🔄 Cell Tiếp Tục Khi Bị Crash (Resume)

In [ ]:
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO

RESUME_CHECKPOINT = f"{RUNS_DIR}/cbam_backbone/train/weights/last.pt"

if os.path.exists(RESUME_CHECKPOINT):
    print(f"🔄 Đang tiếp tục huấn luyện từ: {RESUME_CHECKPOINT}")
    model = YOLO(RESUME_CHECKPOINT)
    model.train(resume=True)
else:
    print(f"❌ Không tìm thấy checkpoint: {RESUME_CHECKPOINT}")